In [29]:
import pandas as pd

In [30]:
df = pd.read_csv("IMDB_dataset.csv")

In [31]:
#  Checking for null values

df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [32]:
# Removing the duplicate values
df.drop_duplicates(inplace = True)
df.shape

(49582, 2)

# Text Pre-Processing

## 1. Lowercasing

In [33]:
df["review"] = df["review"].str.lower()

## 2. Removing URLs

In [34]:
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text) # (pattern, repl, string )
    return text
    # r is for regular expression. It is the syntax
    # \S is for non-white space characters after a certain string which is http here
    # \s is for white spaces like tab, single or double space etc.

df["review"] = df["review"].apply(remove_urls)

In [ ]:
df["review"] = df["review"].apply(remove_html_tags)


## 3. Removing Punctuations

In [35]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)  # we want to keep only A-Z, a-z, 0-9, \s
                                                # the ^ symbol means to exclude whatever is ahead of it.
    return text

df["review"] = df["review"].apply(remove_punctuations)

## 4. Removing HTML tags

In [36]:
def remove_html_tags(text):
    text = re.sub(r"<.*?>", "", text)  # This will remove text inside <> from file
    return text                      # . means all characters
                                     # * means 0 or more characters
                                     # ? means in a non greedy manner, means as soon as we get the closing tag >, stop
                                     # <.*?> is the sequence mimicking the html tag
df["review"] = df["review"].apply(remove_html_tags) 

## 5. Removing Stop Words

In [27]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/swapnil/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/swapnil/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/swapnil/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [37]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [40]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    filtered = [w for w in tokens if w not in stop_words]
    return " ".join(filtered)

df["review"] = df["review"].apply(remove_stopwords)


In [41]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


## 6. Stemming

In [42]:
# changing to root word
# we use a populare stemming alog - PorterStemming

from nltk.stem import PorterStemmer

In [43]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)  

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [44]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


## 7. Encoding the Labels

In [46]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [69]:
y = df["sentiment"]
# this will be used for splitting the data in train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["review"], y, test_size=0.2, random_state=42
)


## 8. Vectorization ( convert text to numbers via TF-IDF)

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X_train = tf.fit_transform(X_train_raw)
X_test = tf.transform(X_test_raw)


In [54]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057107 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515418152739708
  (0, 2868)	0.09361069893377376
  (0, 4940)	0.11467172046351552
  (0, 3002)	0.4719997055075461
  (0, 1274)	0.1357682542446472
  (0, 2288)	0.04949964043885377
  (0, 1932)	0.07912507640921404
  (0, 3550)	0.09639627026026701
  (0, 1361)	0.06162415044555843
  (0, 1962)	0.06155970244220599
  (0, 219)	0.08588817394567325
  (0, 1619)	0.07381616465711817
  (0, 4369)	0.04199368115713442
  (0, 4171)	0.17799470700204995
  (0, 3693)	0.03353179370294512
  (0, 4737)	0.26798619671584006
  (0, 3805)	0.044275563779471706
  (0, 4769)	0.058766745500469335
  (0, 1738)	0.03752043132915078
  (0, 4496)	0.0761397449726664
  (0, 3857)	0.17539536397008615
  (0, 1629)	0.0614237138086635
  (0, 1861)	0.07433044917548241
  (0, 3329)	0.06406741228509
  (0, 3332)	0.084474449303005
  :	:
  (49581, 4890)	0.10682287060750466
  (49581, 1541)	0.1758399379957104
  (4

# 3. Dataset & Data Loaders

In [61]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [58]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [66]:
# We don't use sparse matrices for Dataloaders, so we have to convert these into numpy array
# if these are not already a numpy array.

In [67]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

# torch.from_numpy() is a function used to convert numpy arrays to tensors.

/var/folders/yq/61j601wx2psbghm1wpmrycxm0000gn/T/ipykernel_2958/1903206697.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  torch.from_numpy(y_train.values).float()


In [68]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle = True, batch_size = 64)

# We are using shuffle in test_loader as the test set also has 10k values, which is large.

# 4. Build The RNN

In [70]:
import torch.nn as nn
import torch.optim as optim

In [71]:
class RNN(nn.Module): # Many to one architecture
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional step => It is initializing the h0 value as zero, by default also it is the same
        # the shape of h0 => (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden_size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        # very similar to a probability value
        return out

In [72]:
input_size = X_train.shape[1]

# Creating the model object
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training the RNN

In [74]:
epochs = 20

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton dimension

        outputs = model(Xb)  # it has the dimension (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze())# (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward()  # backprop
        optimizer.step() # update the weights

    print(f"epoch = {epoch+1}/{epochs} loss = {loss.item()}")

epoch = 1/20 loss = 0.13022762537002563
epoch = 2/20 loss = 0.3347226083278656
epoch = 3/20 loss = 0.3092404901981354
epoch = 4/20 loss = 0.3247431814670563
epoch = 5/20 loss = 0.13516204059123993
epoch = 6/20 loss = 0.2193727344274521
epoch = 7/20 loss = 0.19767990708351135
epoch = 8/20 loss = 0.24797073006629944
epoch = 9/20 loss = 0.2974957525730133
epoch = 10/20 loss = 0.2708170413970947
epoch = 11/20 loss = 0.1774812638759613
epoch = 12/20 loss = 0.2771942615509033
epoch = 13/20 loss = 0.26529988646507263
epoch = 14/20 loss = 0.16220276057720184
epoch = 15/20 loss = 0.17870014905929565
epoch = 16/20 loss = 0.19522680342197418
epoch = 17/20 loss = 0.5405720472335815
epoch = 18/20 loss = 0.1626219004392624
epoch = 19/20 loss = 0.2085099071264267
epoch = 20/20 loss = 0.3197307586669922


In [77]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted ==yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals * 100}")

accuracy = 85.57023293334677
